In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# =====================================
# LOAD DATA
# =====================================

df = pd.read_csv("resume_dataset_200k_enhanced.csv")

print("Original Shape:", df.shape)

report = {}

Original Shape: (200000, 17)


In [2]:
# =====================================
# AGE VALIDATION
# =====================================

df = df[(df["age"] >= 16) & (df["age"] <= 65)]

In [3]:
# =====================================
# CGPA CLEANING
# =====================================

report["invalid_cgpa"] = int(
    ((df["cgpa"] < 0) | (df["cgpa"] > 10)).sum()
)

df["cgpa"] = df["cgpa"].clip(0, 10)

In [4]:
# =====================================
# EDUCATION ↔ AGE VALIDATION
# =====================================

min_age_map = {
    "High School": 18,
    "Associate": 20,
    "Bachelors": 21,
    "Masters": 23,
    "PhD": 27
}

df["min_age_required"] = (
    df["education_level"]
    .map(min_age_map)
)

report["age_education_violation"] = int(
    (df["age"] < df["min_age_required"]).sum()
)

df = df[
    df["age"] >= df["min_age_required"]
]

In [5]:
# =====================================
# REALISTIC EXPERIENCE RECONSTRUCTION
# =====================================

graduation_age_map = {
    "High School": 18,
    "Associate": 20,
    "Bachelors": 22,
    "Masters": 24,
    "PhD": 28
}

df["graduation_age"] = (
    df["education_level"]
    .map(graduation_age_map)
)

df["max_possible_exp"] = (
    df["age"] -
    df["graduation_age"]
)

df["max_possible_exp"] = (
    df["max_possible_exp"]
    .clip(lower=0)
)

np.random.seed(42)

df["experience_years"] = np.minimum(
    df["experience_years"],
    df["max_possible_exp"]
)

In [6]:
# =====================================
# NEGATIVE VALUE CHECK
# =====================================

count_columns = [
    "internships",
    "projects",
    "programming_languages",
    "certifications",
    "hackathons",
    "research_papers"
]

for col in count_columns:
    df[col] = df[col].clip(lower=0)

In [7]:
# =====================================
# SKILL SCORE VALIDATION
# =====================================

df["skills_score"] = (
    df["skills_score"]
    .clip(0, 100)
)

df["soft_skills_score"] = (
    df["soft_skills_score"]
    .clip(0, 100)
)

In [8]:
# =====================================
# RESUME LENGTH VALIDATION
# =====================================

df["resume_length_words"] = (
    df["resume_length_words"]
    .clip(50, 2000)
)

In [9]:
# =====================================
# DISTRIBUTION CALIBRATION
# Winsorize top 1%
# =====================================

winsor_cols = [
    "internships",
    "projects",
    "programming_languages",
    "certifications",
    "hackathons",
    "research_papers",
    "experience_years"
]

for col in winsor_cols:

    upper = df[col].quantile(0.99)

    df[col] = np.where(
        df[col] > upper,
        upper,
        df[col]
    )

In [10]:
# =====================================
# TECHNICAL STRENGTH
# =====================================

df["technical_strength"] = (
      0.25 * df["programming_languages"]
    + 0.25 * df["projects"]
    + 0.20 * df["certifications"]
    + 0.15 * df["internships"]
    + 0.10 * df["hackathons"]
    + 0.05 * df["research_papers"]
)

In [11]:
# =====================================
# NORMALIZATION
# =====================================

score_features = [
    "cgpa",
    "experience_years",
    "skills_score",
    "soft_skills_score",
    "technical_strength"
]

scaler = MinMaxScaler()

norm_cols = [
    f"{col}_norm"
    for col in score_features
]

df[norm_cols] = scaler.fit_transform(
    df[score_features]
)

In [12]:
# =====================================
# UNIVERSITY TIER BONUS
# =====================================

df["tier_score"] = 0

df.loc[
    df["university_tier"] == "Tier 1",
    "tier_score"
] = 1

df.loc[
    df["university_tier"] == "Tier 2",
    "tier_score"
] = 0.5

/tmp/ipykernel_17878/755365931.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[


In [13]:
# =====================================
# RESEARCH-BASED HIRING SCORE
# =====================================

df["hiring_score"] = (
      0.30 * df["skills_score_norm"]
    + 0.25 * df["experience_years_norm"]
    + 0.20 * df["technical_strength_norm"]
    + 0.10 * df["soft_skills_score_norm"]
    + 0.10 * df["cgpa_norm"]
    + 0.05 * df["tier_score"]
)

In [14]:
# =====================================
# HUMAN DECISION NOISE
# =====================================

noise = np.random.normal(
    0,
    0.05,
    len(df)
)

df["hiring_score"] = (
    df["hiring_score"] +
    noise
)

In [15]:
# =====================================
# CREATE NEW LABEL
# Top 30% Hired
# =====================================

threshold = (
    df["hiring_score"]
    .quantile(0.70)
)

df["hired_new"] = (
    df["hiring_score"] >= threshold
).astype(int)

In [16]:
# =====================================
# VALIDATION OUTPUT
# =====================================

print("\n========== VALIDATION ==========")

print(
    "Age vs Experience:",
    round(
        df["age"].corr(
            df["experience_years"]
        ),
        3
    )
)

print(
    "Experience vs Hired_New:",
    round(
        df["experience_years"].corr(
            df["hired_new"]
        ),
        3
    )
)

print(
    "Skills vs Hired_New:",
    round(
        df["skills_score"].corr(
            df["hired_new"]
        ),
        3
    )
)

print(
    "Technical Strength vs Hired_New:",
    round(
        df["technical_strength"].corr(
            df["hired_new"]
        ),
        3
    )
)

print(
    "CGPA vs Hired_New:",
    round(
        df["cgpa"].corr(
            df["hired_new"]
        ),
        3
    )
)

print("\n========== LABEL DISTRIBUTION ==========")
print(
    df["hired_new"]
    .value_counts(normalize=True)
)


========== VALIDATION ==========
Age vs Experience: 0.328
Experience vs Hired_New: 0.42
Skills vs Hired_New: 0.481
Technical Strength vs Hired_New: 0.476
CGPA vs Hired_New: 0.094

========== LABEL DISTRIBUTION ==========
hired_new
0    0.700001
1    0.299999
Name: proportion, dtype: float64


In [17]:
# =====================================
# CLEANUP
# =====================================

drop_cols = [
    "min_age_required",
    "graduation_age",
    "max_possible_exp"
]

df = df.drop(
    columns=drop_cols,
    errors="ignore"
)

In [18]:
df.to_csv(
    "resume_dataset_research_ready.csv",
    index=False
)

print("\nFinal Shape:", df.shape)
print("Saved: resume_dataset_research_ready.csv")


Final Shape: (187244, 26)
Saved: resume_dataset_research_ready.csv


In [19]:
print(df["hired_new"].value_counts(normalize=True))

hired_new
0    0.700001
1    0.299999
Name: proportion, dtype: float64


In [20]:
corrs = df[[
    "experience_years",
    "skills_score",
    "technical_strength",
    "soft_skills_score",
    "cgpa",
    "hired_new"
]].corr()

print(corrs["hired_new"].sort_values(ascending=False))

hired_new             1.000000
skills_score          0.481034
technical_strength    0.475642
experience_years      0.419764
soft_skills_score     0.207403
cgpa                  0.094416
Name: hired_new, dtype: float64


In [21]:
print("Original hired")
print(df["hired"].value_counts())

print("\nNew hired")
print(df["hired_new"].value_counts())

print("\nAgreement")
print((df["hired"] == df["hired_new"]).mean())

Original hired
hired
1    132244
0     55000
Name: count, dtype: int64

New hired
hired_new
0    131071
1     56173
Name: count, dtype: int64

Agreement
0.43819294610241183


In [22]:
print(df["hired_new"].isna().sum())

for col in [
    "skills_score_norm",
    "experience_years_norm",
    "technical_strength_norm",
    "soft_skills_score_norm",
    "cgpa_norm",
    "tier_score",
    "hiring_score",
    "hired_new"
]:
    print(col, df[col].isna().sum())

0
skills_score_norm 0
experience_years_norm 0
technical_strength_norm 0
soft_skills_score_norm 0
cgpa_norm 0
tier_score 0
hiring_score 0
hired_new 0
